In [1]:
# autoreload
%load_ext autoreload
%autoreload 2

In [2]:
import os

# Set GPU to 0
# os.environ["CUDA_VISIBLE_DEVICES"]="0"

import sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from MIMIC_IV_HAIM_API import split_note_document, get_biobert_embeddings

In [3]:
mm_dir = "/mnt/data/yihua/master/datasets/mimic-iv_processed"
output_dir = os.path.join(mm_dir, "preprocessing")

rad_notes_df = pd.read_pickle(os.path.join(output_dir, "notes_text.pkl"))

In [4]:
icu_rad_notes_df = rad_notes_df[rad_notes_df['stay_id'].notna()]

In [5]:
from tqdm import tqdm

icu_rad_notes_df['biobert_embeddings'] = None
for index, row in tqdm(icu_rad_notes_df.iterrows(), total=icu_rad_notes_df.shape[0]):
    curr_subject_id = int(row['subject_id'])
    curr_note_id =row['note_id']
    curr_text = row['text']

    chunk_parse, chunk_length = split_note_document(curr_text, 15)

    embeddings = []
    for chunk in chunk_parse:
        curr_embeddings, _ = get_biobert_embeddings(chunk)
        embeddings.append(curr_embeddings)
    
    icu_rad_notes_df.at[index, 'biobert_embeddings'] = embeddings



100%|██████████| 282788/282788 [1:35:33<00:00, 49.32it/s]  


In [6]:
icu_rad_notes_df.to_pickle(os.path.join(output_dir, "icu_notes_text_embeddings.pkl"))